In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import json
import pathlib
import dotenv
import pickle
import sys
import json5
import csv
import os
import pandas as pd
import numpy as np
from typing import Dict, List
from ast import literal_eval
from datetime import datetime
from pydantic import BaseModel, Field, ValidationError, parse_obj_as

current_dir = pathlib.Path().parent.resolve()
sys.path.append(str(current_dir.parent))

from loguru import logger
# from datahub_integrations.gen_ai.bedrock import BedrockModel
from datahub_integrations.gen_ai.term_suggestion_v2 import get_term_recommendations
from datahub_integrations.gen_ai.description_v2 import (
    extract_metadata_for_urn,
    transform_table_info_for_llm,
)
from datahub_integrations.gen_ai.term_suggestion_v2_context import (
    fetch_glossary_info,
    GlossaryUniverseConfig,
)

from docs_generation.graph_helper import create_datahub_graph
from helper import write_llm_output_to_csv

from term_suggestion_analysis_helper import (
    get_table_and_column_infos_dict, 
    get_failed_response_table_urns, 
    read_labeled_column_data,
    get_prediction_df,
    filter_predictions_df,
    get_classification_report_df
)
    
# TERM_SUGGESTION_GENERATION_MODEL: BedrockModel = parse_obj_as(
#     BedrockModel,
#     os.getenv(
#         "TERM_SUGGESTION_GENERATION_BEDROCK_MODEL", BedrockModel.CLAUDE_3_HAIKU.value
#     ),
# )

dotenv.load_dotenv(current_dir / ".env")

True

In [4]:
experiment_dict = [
    {
        "prompt_path": "prompts/Claude_4_no_example.txt", 
        "experiment_name": "Claude_4_no_example", 
        "iterations": 3, 
        "confidence_thresholds": [1,2,3,4]
    },
    {
        "prompt_path": "prompts/Claude_4.txt", 
        "experiment_name": "Claude_4", 
        "iterations": 3, 
        "confidence_thresholds": [1,2,3,4]
    },
    {
        "prompt_path": "prompts/Claude_10_no_example.txt", 
        "experiment_name": "Claude_10_no_example", 
        "iterations": 3, 
        "confidence_thresholds": [5,6,7,8,8.5,9,9.5,10]
    },
    {
        "prompt_path": "prompts/Default_10.txt", 
        "experiment_name": "Default_10", 
        "iterations": 3, 
        "confidence_thresholds": [5,6,7,8,8.5,9,9.5,10]
    },
    {
        "prompt_path": "prompts/Default_4.txt", 
        "experiment_name": "Default_4", 
        "iterations": 3, 
        "confidence_thresholds": [1,2,3,4]
    },
    {
        "prompt_path": "prompts/Claude_10.txt", 
        "experiment_name": "Claude_10", 
        "iterations": 3, 
        "confidence_thresholds": [5,6,7,8,8.5,9,9.5,10]
    },
]

In [5]:
BASE_DIRECTORY = current_dir / f"Tests_{datetime.now().strftime('%m-%d')}/"
COLUMN_LABELS_CSV_PATH = current_dir / "column_labels_without_care_data_updated.csv"
URNS_DICT_PATH = current_dir / "test_urns.json"
INSTANCES_TO_EXAMINE = ['longtailcompanions', 'acryl', 'chime', 'notion', 'twilio']
GLOSSARY_INSTANCE = "longtailcompanions"
GLOSSARY_NODES = ["urn:li:glossaryNode:PIITerms"]
GLOSSARY_TERMS = []
URNS_DICT: Dict[str, List[str]] = json5.loads(  # type: ignore
    (URNS_DICT_PATH).read_text()
)
FILTERS = [
    "no_filter", 
    "description", 
    "sample_values", 
    "description_and_sample_values", 
    "description_or_sample_values", 
    "name_and_datatype",
]
URNS_DICT = {key:value for key, value in URNS_DICT.items() if key in INSTANCES_TO_EXAMINE}
# CONFIDENCE_THRESHOLDS = [1,2,3,4] #[7, 7.5, 8, 8.5, 9, 9.5] # [1,2,3,4,5] #
# URND_DICT = {"longtailcompanions": URNS_DICT["longtailcompanions"][0]}


In [6]:
tables_info_dict, columns_info_dict = get_table_and_column_infos_dict(urns_dict=URNS_DICT)

urn:li:dataset:(urn:li:dataPlatform:snowflake,datahub_community.orbit_fivetran.member_in_workspace,PROD)
urn:li:dataset:(urn:li:dataPlatform:snowflake,datahub_community.orbit.workspace_activity,PROD)
urn:li:dataset:(urn:li:dataPlatform:snowflake,analytics.looker.zendesk_agent,PROD)
urn:li:dataset:(urn:li:dataPlatform:looker,snowflake.view.penny_claim_created,PROD)
urn:li:dataset:(urn:li:dataPlatform:snowflake,analytics.omx.disputes_omxri,PROD)
urn:li:dataset:(urn:li:dataPlatform:snowflake,analytics.looker.user_socure_cip_dim,PROD)
urn:li:dataset:(urn:li:dataPlatform:snowflake,analytics.omx.five9_call_acd_sla,PROD)
urn:li:dataset:(urn:li:dataPlatform:snowflake,analytics.omx.five9_call_acd_sla_view,PROD)
urn:li:dataset:(urn:li:dataPlatform:snowflake,postgres_db.mms.external_card_transfers,PROD)
urn:li:dataset:(urn:li:dataPlatform:snowflake,edw_db.core.fct_settled_transaction,PROD)
urn:li:dataset:(urn:li:dataPlatform:snowflake,production.core.stripe_customer_daily,PROD)
urn:li:dataset:(ur

Ignoring unknown aspect type entityInferenceMetadata
Ignoring unknown aspect type entityInferenceMetadata


urn:li:dataset:(urn:li:dataPlatform:snowflake,long_tail_companions.adoption.human_profiles,PROD)
urn:li:dataset:(urn:li:dataPlatform:snowflake,long_tail_companions.adoption.pets,PROD)
urn:li:dataset:(urn:li:dataPlatform:snowflake,long_tail_companions.analytics.active_customer_ltv,PROD)
urn:li:dataset:(urn:li:dataPlatform:snowflake,long_tail_companions.analytics.dog_breeds,PROD)
urn:li:dataset:(urn:li:dataPlatform:looker,analytics.explore.retention_cohort,PROD)
urn:li:dataset:(urn:li:dataPlatform:looker,long_tail_companions.view.pet_details,PROD)
urn:li:dataset:(urn:li:dataPlatform:looker,long_tail_companions.explore.monthly_adoption_projection,PROD)
urn:li:dataset:(urn:li:dataPlatform:redshift,banking.public.account_transactions,PROD)


In [57]:
# prompt = (current_dir / "../../src/datahub_integrations/gen_ai/term_suggestion_prompt.txt").read_text()
# prompt

# Fetch Glossary terms

In [7]:
graph_client = create_datahub_graph(GLOSSARY_INSTANCE)
glossary_config = GlossaryUniverseConfig(
    glossary_nodes=GLOSSARY_NODES,
    glossary_terms=GLOSSARY_TERMS
)
glossary_info = fetch_glossary_info(graph_client=graph_client, universe=glossary_config)

print(len(glossary_info.glossary.keys()))
terms_to_remove = ['Day', 'Month', 'Year']
glossary_info.glossary = {key: value for key, value in glossary_info.glossary.items() if value["term_name"] not in terms_to_remove}
print(len(glossary_info.glossary.keys()))

49
46


In [59]:
terms = []
for key, value in glossary_info.glossary.items():
    terms.append(value["term_name"])
print(terms)

['Username', 'SSN', 'PassportNumber', 'CreditCardNumber', 'TwitterHandle', 'Nationality', 'BankRoutingNumber', 'Product ID', 'Year', 'BankAccountNumber', 'PhoneNumber', 'FullName', 'BillingAddress', 'CreditCardType', 'MedicalRecordNumber', 'WorkEmail', 'JobTitle', 'DeviceID', 'Age', 'FirstName', 'EmailAddress', 'CountryOfResidence', 'HomeAddress', 'LinkedInProfile', 'BiometricData', 'TaxID', 'EmploymentStatus', 'MACAddress', 'LoginHistory', 'FacebookProfile', 'BrowsingHistory', 'IPAddresses', 'LastName', 'PasswordHash', 'MiddleName', 'ShippingAddress', 'GenderIdentity', 'InstagramHandle', 'CreditCardExpirationDate', 'DriverLicenseNumber', 'Cookies', 'HealthInsuranceID', 'WorkPhone', 'EmployerName', 'CVV', 'BirthDate', 'Country Code', 'Day', 'Month']


49
46


In [61]:
# urns_dict_ = {key: value for key, value in urns_dict.items() if key in ['longtailcompanions', 'acryl']}

In [19]:
def func_categorize(row):
    if len(row["pred_max_score_term"])==0:
        if row["glossary_terms_updated"] is None:
            return "match-no_assignment"
        else:
            return "mismatch-actual_only"
    elif row["glossary_terms_updated"] is None:
            return "mismatch-predicted_only"
    else:
        actual_terms = [term.strip() for term in row["glossary_terms_updated"].split("/")]
        if len(np.intersect1d(actual_terms, row["pred_max_score_term"])) == 0:
            return "mismatch"
        else:
            return "match-term_assigned"
        

def check_match(row):
    #     print(row)
    if (
        (row["glossary_term"] is None and len(row["pred_max_score_term"]) == 0)
        or row["glossary_term"] in row["pred_max_score_term"]
        or row["Alternate_Glossary_Or_Comment"] in row["pred_max_score_term"]
    ):
        return 1
    else:
        return 0

def get_merged_prediction_df(labeled_df, df_pred):
    merged_df = pd.merge(
        labeled_df,
        df_pred,
        on="unique_keys",
        how="left",
    )
    merged_df.loc[:, "pred_max_score_term"] = merged_df["pred_max_score_term"].apply(
        lambda x: x if isinstance(x, list) else []
    )
    merged_df.loc[:, "predicted_labels"] = merged_df["predicted_labels"].apply(
        lambda x: x if isinstance(x, list) else []
    )
    not_omitted_columns_df = merged_df[
        merged_df.unique_keys.isin(df_pred.unique_keys.tolist())
    ]
    omitted_columns_df = merged_df[
        ~merged_df.unique_keys.isin(df_pred.unique_keys.tolist())
    ]
    
#     merged_df.loc[:, "match"] = merged_df.apply(lambda x: check_match(x), axis=1)
    merged_df.loc[:, "label_class"] = merged_df.apply(lambda x: func_categorize(x), axis=1)
    return merged_df, not_omitted_columns_df, omitted_columns_df

def get_parsed_responses_for_single_experiment_run(urns_dict, glossary_info, prompt_path):
    raw_llm_responses = []
    parsed_llm_responses = []

    for instance, urns in urns_dict.items():
        graph_client = create_datahub_graph(instance)
        for urn in urns:
            print(urn)
            try:
                table_terms, column_terms, raw_llm_response = get_term_recommendations(
                    table_urn=urn, graph_client=graph_client, glossary_info=glossary_info, prompt_path = prompt_path
                )
                #             column_terms = label_fake_terms(column_terms)
                raw_llm_responses.append([urn, instance, raw_llm_response])
                parsed_llm_responses.append((urn, instance, table_terms, column_terms))
            except Exception as e:
                logger.exception(f"Exception Occurred {e}")
                raw_llm_responses.append([urn, instance, raw_llm_response])
                parsed_llm_responses.append((urn, instance, None, None))
                continue
    return parsed_llm_responses, raw_llm_responses

def get_aggregated_metrics(classification_report_df, confidence_threshold):
    total_TP = classification_report_df.TP.sum()
    total_FP1 = classification_report_df.FP1.sum()
    total_FP2 = classification_report_df.FP2.sum()
    total_FN = classification_report_df.FN.sum()
    total_TN = classification_report_df.TN.sum()

    precision_term_for_all_tables = total_TP / (
        total_TP + total_FP1 + total_FP2
    ) if (total_TP + total_FP1 + total_FP2) != 0 else np.nan

    recall_term_for_all_tables = total_TP / (
        total_TP + total_FN + total_FP2
    ) if (total_TP + total_FN + total_FP2) != 0 else np.nan

    precision_none_for_all_tables = total_TN / (
        total_TN + total_FN
    ) if (total_TN + total_FN) != 0 else np.nan

    recall_none_for_all_tables = total_TN / (
        total_TN + total_FP1
    ) if (total_TN + total_FP1) != 0 else np.nan

    
    aggregated_metrics = {
        "precision_term_for_all_tables": precision_term_for_all_tables,
        "recall_term_for_all_tables": recall_term_for_all_tables,
        "precision_none_for_all_tables": precision_none_for_all_tables,
        "recall_none_for_all_tables": recall_none_for_all_tables,
        "threshold": confidence_threshold,
    }
    return aggregated_metrics
                

def get_final_statistics_for_confidence_threshold(
    merged_prediction_df, 
    parsed_llm_responses, 
    filters,
    columns_info_dict,
    labeled_df,
    confidence_threshold,
):
    final_stats = {}
    classification_reports = {}
    for filter_ in filters:
        filtered_prediction_df = filter_predictions_df(merged_prediction_df, filter_)
        classification_report_df = get_classification_report_df(
            parsed_llm_responses=parsed_llm_responses,
            pred_df=filtered_prediction_df,
            column_infos_dict=columns_info_dict,
            labeled_df=labeled_df,
            filter_=filter_,
        )
        classification_reports[filter_] = classification_report_df
        temp_stats = dict(
            classification_report_df[
                [
                    "fake_columns_count",
                    "fake_terms_count",
                    "actual_column_count",
                    "skipped_columns_count",
                    "TP",
                    "TN",
                    "FP1",
                    "FP2",
                    "FN",
                ]
            ].sum(axis=0)
        )
        aggregated_metrics = get_aggregated_metrics(
            classification_report_df=classification_report_df, 
            confidence_threshold=confidence_threshold
        )
        temp_stats.update(aggregated_metrics)
        final_stats[filter_] = temp_stats  
    final_stats_df = pd.DataFrame(final_stats)
    return final_stats_df, classification_reports

def save_misclassification_analysis(merged_prediction_df, dest_dir, confidence_threshold):
    miscalssification_analysis_cols = [
            "table_urn",
            "col_name",
            "col_description",
            "sample_values",
            "glossary_terms_updated",
            "Alternate_Glossary_Or_Comment",
            "predicted_labels",
    ]

    merged_prediction_df[merged_prediction_df.label_class == "mismatch-actual_only"][
        miscalssification_analysis_cols
    ].to_csv(dest_dir / f"FN_threshold_{confidence_threshold}.csv")

    merged_prediction_df[merged_prediction_df.label_class == "mismatch-predicted_only"][
        miscalssification_analysis_cols
    ].to_csv(dest_dir / f"FP1_threshold_{confidence_threshold}.csv")

    merged_prediction_df[merged_prediction_df.label_class == "mismatch"][miscalssification_analysis_cols].to_csv(
        dest_dir / f"FP2_threshold_{confidence_threshold}.csv"
    )
    return None

def populate_analysis_for_confidence_threshold_list(
    output_dir,
    parsed_llm_responses,
    filters,
    urns_dict,
    columns_info_dict,
    confidence_thresholds,
):
    for confidence_threshold in confidence_thresholds:
        print("CONFIDENCE_THRESHOLD", confidence_threshold)

        # Make Directory
        SUB_DIR = output_dir / f"threshold_{confidence_threshold}"
        SUB_DIR.mkdir(parents=True, exist_ok=True)

        # Read labelled data
        labeled_df = read_labeled_column_data(COLUMN_LABELS_CSV_PATH, urns_dict)
        tables_with_parsing_error, skipped_tables = get_failed_response_table_urns(
            parsed_llm_responses
        )
        print("tables_with_parsing_error:", tables_with_parsing_error)
        print("skipped_tables:", skipped_tables)

        # Prepare prediction df:
        df_pred = get_prediction_df(parsed_llm_responses, confidence_threshold)
        print("labeled_df", len(labeled_df))
        print("prediction_df", len(df_pred))

        # Merge Prediction df and labelled df
        merged_df, _, _ = get_merged_prediction_df(
            labeled_df[~labeled_df.table_urn.isin(tables_with_parsing_error)], 
            df_pred,
        )
        print("merged_df", len(merged_df))
        merged_df.to_csv(
            SUB_DIR / f"predicted_labels_threshold_{confidence_threshold}.csv"
        )

        # Misclassification Analysis:
        save_misclassification_analysis(merged_df, SUB_DIR, confidence_threshold)

        # Final Statistics
        final_stats_df, classification_reports = get_final_statistics_for_confidence_threshold(
            merged_prediction_df=merged_df, 
            parsed_llm_responses=parsed_llm_responses, 
            filters=filters,
            columns_info_dict=columns_info_dict,
            labeled_df=labeled_df,
            confidence_threshold=confidence_threshold
        )
        
        for key, classification_report_df in classification_reports.items():
            classification_report_df.to_csv(SUB_DIR / f"{key}_cf_df.csv")
        final_stats_df.to_csv(SUB_DIR / f"final_stats_threshold_{confidence_threshold}.csv") 
    return None


In [20]:
from tqdm import tqdm

In [ ]:
# TODO: Restructure the following block to make it more modular

for experiment in tqdm(experiment_dict, total=len(experiment_dict)):
    (BASE_DIRECTORY / f"{experiment["experiment_name"]}").mkdir(parents=True, exist_ok=True)
    for run in range(experiment["iterations"]):
        print(f"run_{run}")
        # Create Output Directory:
        OUTPUT_DIR = BASE_DIRECTORY / f"{experiment["experiment_name"]}" / f"run{run+1}_{datetime.now().strftime('%H-%M-%S')}"
        OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
        # read and save prompt:
        prompt_path = current_dir / experiment["prompt_path"]
        prompt = prompt_path.read_text()
        pathlib.Path(OUTPUT_DIR / "prompt.txt").write_text(prompt)
        
        # Generate Response:
        parsed_llm_responses, raw_llm_responses = get_parsed_responses_for_single_experiment_run(
            urns_dict=URNS_DICT, 
            glossary_info=glossary_info,
            prompt_path=prompt_path
        )
        
        # Write response to directory
        write_llm_output_to_csv(llm_response=parsed_llm_responses, csv_path=OUTPUT_DIR / "parsed_response.csv")
        with open(OUTPUT_DIR / "parsed_response.pkl", "wb") as fp:
            pickle.dump(parsed_llm_responses, fp)
            fp.close()
         
        # Populate Analysis
        populate_analysis_for_confidence_threshold_list(
            output_dir=OUTPUT_DIR,
            confidence_thresholds=experiment["confidence_thresholds"],
            filters=FILTERS,
            urns_dict=URNS_DICT,
            parsed_llm_responses=parsed_llm_responses,
            columns_info_dict=columns_info_dict
        )


  0%|                                                                                            | 0/6 [00:00<?, ?it/s]

run_0
urn:li:dataset:(urn:li:dataPlatform:snowflake,datahub_community.orbit_fivetran.member_in_workspace,PROD)


2024-09-20 18:11:39.769 | DEBUG    | datahub_integrations.gen_ai.term_suggestion_v2:get_term_recommendations_for_column_splits:171 - Column split lengths: [21, 21]
2024-09-20 18:11:39.769 | INFO     | datahub_integrations.gen_ai.term_suggestion_v2:generate_prompt:111 - Reading prompt from C:\PROJECTS_\Acryl\acryl_main_repo\datahub-fork\datahub-integrations-service\experiments\term_suggestion_v1\prompts\Claude_4_no_example.txt
2024-09-20 18:11:53.423 | DEBUG    | datahub_integrations.gen_ai.bedrock:call_bedrock_llm:84 - LLM call took 13.654085874557495 seconds
2024-09-20 18:11:53.431 | INFO     | datahub_integrations.gen_ai.term_suggestion_v2:generate_prompt:111 - Reading prompt from C:\PROJECTS_\Acryl\acryl_main_repo\datahub-fork\datahub-integrations-service\experiments\term_suggestion_v1\prompts\Claude_4_no_example.txt
2024-09-20 18:11:58.443 | DEBUG    | datahub_integrations.gen_ai.bedrock:call_bedrock_llm:84 - LLM call took 5.012205362319946 seconds


urn:li:dataset:(urn:li:dataPlatform:snowflake,datahub_community.orbit.workspace_activity,PROD)


2024-09-20 18:11:59.502 | DEBUG    | datahub_integrations.gen_ai.term_suggestion_v2:get_term_recommendations_for_column_splits:171 - Column split lengths: [24]
2024-09-20 18:11:59.502 | INFO     | datahub_integrations.gen_ai.term_suggestion_v2:generate_prompt:111 - Reading prompt from C:\PROJECTS_\Acryl\acryl_main_repo\datahub-fork\datahub-integrations-service\experiments\term_suggestion_v1\prompts\Claude_4_no_example.txt
2024-09-20 18:12:13.974 | DEBUG    | datahub_integrations.gen_ai.bedrock:call_bedrock_llm:84 - LLM call took 14.448509931564331 seconds


...

2024-09-20 20:06:18.375 | DEBUG    | datahub_integrations.gen_ai.term_suggestion_v2:get_term_recommendations_for_column_splits:171 - Column split lengths: [40]
2024-09-20 20:06:18.377 | INFO     | datahub_integrations.gen_ai.term_suggestion_v2:generate_prompt:111 - Reading prompt from C:\PROJECTS_\Acryl\acryl_main_repo\datahub-fork\datahub-integrations-service\experiments\term_suggestion_v1\prompts\Claude_4.txt
2024-09-20 20:06:41.034 | DEBUG    | datahub_integrations.gen_ai.bedrock:call_bedrock_llm:84 - LLM call took 22.654279470443726 seconds


urn:li:dataset:(urn:li:dataPlatform:looker,snowflake.view.penny_claim_created,PROD)


2024-09-20 20:06:41.780 | DEBUG    | datahub_integrations.gen_ai.term_suggestion_v2:get_term_recommendations_for_column_splits:171 - Column split lengths: [10]
2024-09-20 20:06:41.782 | INFO     | datahub_integrations.gen_ai.term_suggestion_v2:generate_prompt:111 - Reading prompt from C:\PROJECTS_\Acryl\acryl_main_repo\datahub-fork\datahub-integrations-service\experiments\term_suggestion_v1\prompts\Claude_4.txt
2024-09-20 20:06:48.682 | DEBUG    | datahub_integrations.gen_ai.bedrock:call_bedrock_llm:84 - LLM call took 6.896360158920288 seconds


urn:li:dataset:(urn:li:dataPlatform:snowflake,analytics.omx.disputes_omxri,PROD)


2024-09-20 20:06:49.473 | DEBUG    | datahub_integrations.gen_ai.term_suggestion_v2:get_term_recommendations_for_column_splits:171 - Column split lengths: [28, 28, 26]
2024-09-20 20:06:49.474 | INFO     | datahub_integrations.gen_ai.term_suggestion_v2:generate_prompt:111 - Reading prompt from C:\PROJECTS_\Acryl\acryl_main_repo\datahub-fork\datahub-integrations-service\experiments\term_suggestion_v1\prompts\Claude_4.txt
2024-09-20 20:07:01.760 | DEBUG    | datahub_integrations.gen_ai.bedrock:call_bedrock_llm:84 - LLM call took 12.282447576522827 seconds
2024-09-20 20:07:01.763 | INFO     | datahub_integrations.gen_ai.term_suggestion_v2:generate_prompt:111 - Reading prompt from C:\PROJECTS_\Acryl\acryl_main_repo\datahub-fork\datahub-integrations-service\experiments\term_suggestion_v1\prompts\Claude_4.txt


In [82]:
# OUTPUT_DIR = BASE_DIRECTORY / "output_09-19-2024_13-52-11"
# with open(OUTPUT_DIR / "parsed_response.pkl", 'rb') as f:
#     parsed_llm_responses = pickle.load(f) 
#     f.close()

In [83]:
"EmailAddress / WorkEmail".split("/")

['EmailAddress ', ' WorkEmail']

In [84]:
for x in labeled_df.glossary_terms_updated:
    if x is not None:
        for y in x.split("/"):
            if y.strip() in terms:
                continue
            else:
                print(x)

# Generate Analysis Reports

     **actual**            **predicted**           **cateory_name**
       None                    None                  match-no_assignment              TN
       None                    term                  mismatch-predicted_only          FP1
       term                    different term        mismatch                         FP2
       term                    term                  match-term_assigned              TP
       term                    None                  mismatch-actual_only             FN

    Recall of positive class:  TP/(TP + FN + FP2)
    Precision : TP/(TP + FP1 + FP2)

    Recal of negative class: TN/(TN + FP1)
    Precision of negative class: TN/(TN + FN)


In [21]:
# no assignment- negative
# assignment - positive